# 2 · Petrological Data Handling with `petropandas`

**Module 1 · Data Processing with `petropandas`, part 2**

`petropandas` extends every pandas `DataFrame` with petrology-aware
"accessors" — `.oxides`, `.moles`, `.cations`, `.mineral` and `.bulk` — that
know about oxide chemistry, mineral structural formulas, and bulk-rock
recalculation. They sit *on top of* the pandas you learned in Notebook 1: a
`petropandas`-enhanced DataFrame is still a completely ordinary DataFrame.

Sections 1–6 use datasets bundled with `petropandas` itself (zero external
files). Sections 7–8 switch to **this workshop's own dataset**,
`data/avg_pelite/microprobe_data.xlsx` — real EPMA data for a metapelite
sample that was used to build the P–T pseudosection you'll explore in
`03_psexplorer.ipynb` this afternoon. That's the thread running through the
whole day: raw microprobe data → mineral formulas → bulk composition → P–T
path.


## 2.1 Introduction to `petropandas`

`from petropandas import pd, ...` re-exports pandas itself, plus every
built-in `Mineral` model (`Grt`, `Cpx`, `Amp`, ...) as ready-to-use objects —
so a single import gives you pandas *and* petrology.

We load three bundled datasets:
- `minerals` — the same 315-analysis, 21-mineral-group dataset from
  Notebook 1.
- `grt_profile` — a 99-point garnet core-to-rim EPMA traverse.
- `bulk` — 9 bulk-rock/area EPMA spectra.


In [ ]:
from petropandas import pd, Grt, Cpx, Amp, TernaryPlot, ProfilePlot, ScatterPlot
from petropandas.data import minerals, grt_profile, bulk

# Common display options (optional)
pd.set_option('display.max_rows', 10)
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.2f}'.format)

minerals

Forget the exact object name for a mineral? `mdb` is a small registry over
all 16 built-in models — look one up by name or abbreviation instead of
digging through the source:


In [ ]:
from petropandas import mdb

print(mdb.names)
print(mdb.by_name("Garnet") is Grt)
print(mdb.by_abbreviation("cpx") is Cpx)  # case-insensitive


## 2.2 The `.oxides`, `.moles` and `.cations` accessors

`.oxides` cleans up and standardises a DataFrame's oxide columns (handling
column-name aliases like `FeOT` → `FeO`, sorting them into a standard
petrological order). `.moles()` converts oxide wt% to molar proportions, and
`.cations(n_oxygens=...)` converts to **atoms per formula unit (apfu)** on a
given oxygen basis — the fundamental step behind every mineral-formula
calculation in petrology.

Let's pull out the garnet subset and walk through the unit conversions.
`df.oxides.select(...)` is `petropandas`'s filtering helper — equivalent to
the boolean indexing from Notebook 1, but chainable with the other
accessors.


In [ ]:
g = minerals.oxides.select("Garnet", on="Mineral")
g = g.oxides.sorted()
g


In [ ]:
g_moles = g.moles()
g_moles


In [ ]:
# Garnet is conventionally recalculated on a 12-oxygen basis
g_cations = g.cations(n_oxygens=12)
g_cations


You'll notice `petropandas` re-labels cation columns in `Element{charge}` form —
`Si{4+}`, `Fe{2+}`, `Al{3+}`, ... The braces are part of the *column name*: to
select one, quote the whole string, e.g. `cations["Fe{2+}"]`. Labelling the
charge matters because `Fe{2+}` and `Fe{3+}` are different ions with different
sites and end-members — a charge-less `df["Fe"]` would be ambiguous.

You'll meet these site-fraction names again this afternoon: `xFeX` and `xCaX`
on the pseudosection (`03_psexplorer.ipynb`) are exactly the Fe and Ca
fractions of this same garnet `X` site.


Notice the total cations per formula unit should land close to 8 (garnet's
ideal cation count, X3Y2Z3O12 -> 3+2+3 = 8) — a quick internal-consistency
check before trusting the analysis further.


In [ ]:
g_cations.sum(axis=1)


In [ ]:
g_cations.sum(axis=1).describe()


## 2.3 Structural formula recalculation

`df.mineral.apfu(mineral_model)` and `df.mineral.site_allocations(mineral_model)`
do the full job: oxide wt% → cations → allocated onto the mineral's
crystallographic sites — in one call. The **oxygen basis and site model are
not a DataFrame column**: they come from the `Mineral` object you pass in
(`Grt`, `Cpx`, `Opx`, ...), so the same DataFrame code works for any mineral,
you just swap the object.

- **Garnet** (`Grt`): 12 oxygens, sites `Z` (Si,Al) / `Y` (Al,Fe3+) / `X`
  (Fe2+,Mg,Ca,Mn).
- **Clinopyroxene** (`Cpx`): 6 oxygens, a different site model entirely.


In [ ]:
g_site = g.mineral.site_allocations(Grt)
g_site


In [ ]:
cpx = minerals.oxides.select("Clinopyroxene", on="Mineral").oxides()
cpx_apfu = cpx.mineral.apfu(Cpx)  # 6-oxygen basis for pyroxene, no column tells it that -- Cpx does
print("clinopyroxene apfu total (should be close to 4):", cpx_apfu.sum(axis=1).mean().round(3))
cpx_apfu


In [ ]:
g.mineral.apfu(Grt)

In [ ]:
g.mineral.apfu(Grt).cations.calc("XFe", "`Fe{3+}`/(`Fe{3+}` + `Mg{2+}`)")

In [ ]:
g.mineral.site_allocations(Grt)

### Data Quality & Validation
Before processing Electron Probe Microanalysis (EPMA) dataset in `petropandas`, performing data quality checks is essential to prevent analytical artifacts from propagating into downstream thermodynamic modeling. It could involves inspecting total oxide weight percentages, evaluating cation stoichiometry and charge balance limits—such as ensuring structural site occupancy sums and calculated ferric/ferrous iron ratios fall within crystallographically reasonable bounds—provides a second line of defense against poor measurements. Establishing these explicit numerical cutoff guidelines enables the early identification and rejection of flawed probe analyses prior to data import and automated mineral formula normalization.

`df.mineral.check_stoichiometry(mineral_model)` scores an analysis against
several stoichiometric criteria (all close to 1.0 = internally consistent
analysis; lower values flag something worth a second look, e.g. a bad
analytical total or an implausible charge balance).

Columns in the returned DataFrame:

| Metric | Description |
| --- | --- |
| **analytical_total** | Oxide wt% sum vs mineral-specific ideal range. |
| **cation_deviation** | Total APFU vs ideal cation count (NaN if mineral does not define ideal_cations). |
| **charge_balance** | Total positive charge vs expected from oxygen count (exponential decay). |
| **fe3+_validity** | Binary check that $\text{Fe}^{3+}$ and $\text{Fe}^{2+}$ are non-negative after valence splitting (NaN if no Fe split). |
| **site_vacancies** | Mean site occupancy fraction across all sites. |
| **leftover_cations** | Fraction of total APFU not assigned to any site. |
| **tetrahedral_fill** | T-site sum vs T-site capacity (NaN if no T-site defined). |


In [ ]:
g.mineral.check_stoichiometry(Grt)


Six columns is a lot to eyeball for a quick screen of many analyses.
`df.mineral.stoichiometry_quality(mineral_model)` collapses the same check
into a **single 0-1 score per analysis** (the mean of the three
deviation-style criteria above) — handy for sorting/filtering a large batch
before looking at individual `check_stoichiometry` breakdowns.


In [ ]:
g.mineral.stoichiometry_quality(Grt)


In [ ]:
g.mineral.apfu(Grt).cations.calc("X site", "Fe{2+} + Mg{2+} + Ca{2+} + Mn{2+}")

How to find out what cations did not make it into garnet structural formula?

In [ ]:
# drop columns not needed 
g_clean = g.drop(["Na2O", "K2O", "Cr2O3", "ZnO"], axis=1)
# calculate cations on 12 oxygen basis
g_cat = g_clean.cations(n_oxygens=12)
# calculate cations from allocated sites in standard order
g_alloc = g_clean.mineral.apfu(Grt).cations.reframe(g_cat.columns)
# calculate reminder as difference
reminder = g_cat - g_alloc
with pd.option_context('display.float_format', '{:.4f}'.format):
    display(reminder)

## 2.4 End-member calculations

`df.mineral.end_members(mineral_model)` converts the site-allocated formula
into classic mineral end-member percentages — for garnet: Pyrope, Almandine,
Spessartine, Grossular (plus Andradite/Uvarovite for Fe3+/Cr-bearing
garnet).


In [ ]:
g_em = g.mineral.end_members(Grt)
g_em


## 2.5 Visualisation: ternary and profile plots

`petropandas` ships three plotting helpers built for compositional data:
`TernaryPlot`, `ProfilePlot`, `ScatterPlot`. Their axes are simple arithmetic
expressions over DataFrame columns (`"Prp+Sps"` sums two columns on the fly).

First, a classic garnet ternary — Almandine vs (Pyrope+Spessartine) vs
Grossular — for the 99-point core-to-rim `grt_profile` traverse:

To read a ternary: the three named corners are components that add up to
100%, so a single point encodes all three proportions at once (a point on an
edge has zero of the opposite corner's component). The `llim=(35, 100)`,
`tlim=(15, 50)` arguments are just *zoom limits* — they keep the plotted
region readable instead of crushing the data against the diagram corners; for a
first read you can ignore them.

In [ ]:
grt_profile_em = grt_profile.mineral.end_members(Grt)

t = TernaryPlot(
    top="Prp+Sps", left="Alm", right="Grs",
    llim=(35, 100), tlim=(15, 50),
)
t.add(grt_profile_em, label="Grt profile")
t.show()


And a **compositional profile plot** — end-member proportion vs. position
along the traverse, the standard way to visualise garnet zoning:


In [ ]:
p = ProfilePlot(columns=["Alm", "Prp", "Grs", "Sps"], split="auto", title="Garnet zoning profile")
p.add(grt_profile_em, lw=2)
p.show()


## 2.6 Hands-on exercise: unknown amphibole and clinopyroxene analyses

`petropandas.data.minerals` also contains real Amphibole and Clinopyroxene
analyses. Recalculate their structural formulas and end-members with `Amp`
(23 oxygens) and `Cpx` (6 oxygens), and plot how their compositions vary.

1. Select the `Amphibole` rows from `minerals` and clean them with
   `.oxides.sorted()`.
2. Compute `.mineral.end_members(Amp)`.
3. Do the same for `Clinopyroxene` with `Cpx`. Quadrilateral (Quad) component
   could be calculated as 100 - `Jd` - `Ae`
5. Plot the clinopyroxene Quad–Jd–Ae ternary, and a simple amphibole scatter
   of `Tremolite` vs `Pargasite`.

Try it yourself first.


In [ ]:
# Your code here
amp = ...
cpx_unknown = ...

<details><summary><b>Solution (click to expand)</b></summary>

```python
amp = minerals.oxides.select("Amphibole", on="Mineral").oxides.sorted()
amp_em = amp.mineral.end_members(Amp)

cpx_unknown = minerals.oxides.select("Clinopyroxene", on="Mineral").oxides.sorted()
cpx_em = cpx_unknown.mineral.end_members(Cpx)
cpx_em["Quad"] = 100 - cpx_em["Jd"] + cpx_em["Ae"]

t = TernaryPlot(top="Quad", left="Jd", right="Ae")
t.add(cpx_em, label="Unknown Cpx")
t.show()

s = ScatterPlot("Tremolite", "Pargasite")
s.add(amp_em, label="Amp")
s.show()
```

</details>


## 2.8 The `.bulk` accessor and bulk composition for THERMOCALC and MAGEMin

`.bulk` works on whole-rock (or averaged) compositions rather than single
mineral analyses. Beyond cleaning/normalising, it has petrology-specific
diagnostics — here we look at bulk composition formating for thermodynamic softwares.

`data/avg_pelite/microprobe_data.xlsx` has a `Bulk` sheet (the average pelite
bulk composition used to build this afternoon's pseudosection) and a `Grt-profile`
sheet (a real 27-point core-to-rim garnet traverse from that same rock).

**Step 1 — format the bulk composition for THERMOCALC and MAGEMin.**
`.bulk.TCbulk()` and `.bulk.MAGEMin()` take an estimated H2O content and
ferric-iron correction (`oxygen`, THERMOCALC's excess-oxygen convention) and
print/return a ready-to-paste bulk-composition block.


In [ ]:
avgpelite_bulk = pd.read_excel(
    "../data/avg_pelite/microprobe_data.xlsx", sheet_name="Bulk", index_col=0
)
avgpelite_bulk

In [ ]:
# For THERMOCALC copy-paste those two lines into your script file
avgpelite_bulk.bulk.TCbulk(H2O=4, oxygen=0.15)


In [ ]:
# The blocks below should be copy-pasted into `.dat` file later used in MAGEMinApp.
avgpelite_bulk.bulk.MAGEMin(H2O=4, oxygen=0.15)

In [ ]:
# db="mpe" requests a different thermodynamic database than the default "mp";
avgpelite_bulk.bulk.MAGEMin(H2O=4, oxygen=0.15, db="mpe")

**Step 2 — effective bulk composition by fractionating garnet.**

A rock's *bulk* composition is what you'd analyse today — but if garnet grew
progressively and is chemically zoned, earlier-formed garnet interiors are no
longer in equilibrium with the rest of the rock at peak conditions. The
**effective bulk composition** removes a modelled amount of the zoned
mineral (volume-integrated across its own core-to-rim profile) from the bulk,
which is what should really be used for peak-condition modelling.

First, load the real `Grt-profile` sheet and **check the analytical
totals** — this is exactly the kind of EPMA quality check from Notebook 1
section 1.3, and it matters here: totals near the rim/core are close to
100%, but drift upward in the middle of the traverse. We normalise every
row to 100% before using it for fractionation, so a total-drift artifact
doesn't bias the effective bulk composition.


In [ ]:
grt_prof = pd.read_excel("../data/avg_pelite/microprobe_data.xlsx", sheet_name="Grt-profile")
grt_prof

The sheet is already ordered core (row 0, `grt-1-core`) to rim (row 26,
`grt-27-rim`), so we pass `order="core-to-rim"` and fractionate a small
molar fraction of garnet from the bulk (`mineral=Grt` scales `fraction` from
moles of garnet formula units).


In [ ]:
effective_bulk = avgpelite_bulk.bulk.fractionate(
    grt_prof, fraction=0.03, mineral=Grt, order="core-to-rim"
)
comparison = pd.concat(
    [avgpelite_bulk.reset_index(drop=True), effective_bulk.reset_index(drop=True)],
    keys=["Original Bulk", "Effective bulk (3% Grt fractionated)"],
)
comparison


MnO, FeO and CaO drop and MgO/SiO2 shift slightly relative to the whole-rock bulk
— exactly what you'd expect from removing garnet, which strongly partitions
Fe and Ca. This `effective_bulk` (or the un-fractionated `avgpelite_bulk`,
depending on the question being asked) is what would be passed to
`.bulk.TCbulk()`/`.bulk.MAGEMin()` for further pseudosection work.


## Recap

- `.oxides` / `.moles` / `.cations` handle unit conversions; the oxygen/
  cation basis for `.cations()` and the site model for `.mineral.*` come
  from a `Mineral` object (`Grt`, `Cpx`, `Amp`, ...), not a DataFrame column.
- `.mineral.apfu` / `.site_allocations` / `.end_members` /
  `.check_stoichiometry` take you from oxide wt% to a full structural
  formula and end-member breakdown.
- `TernaryPlot` / `ProfilePlot` / `ScatterPlot` plot straight from those
  DataFrames using simple column-arithmetic expressions.
- `.bulk.TCbulk` / `.bulk.MAGEMin` format a bulk composition for
  thermodynamic modelling; `.bulk.fractionate` computes an effective bulk
  composition by removing a zoned mineral's volume-integrated composition.

**Next up:** this afternoon's `03_psexplorer.ipynb` uses `pypsbuilder` to
explore the P–T pseudosection that was built from this same avgpelite bulk
composition — and reuses this notebook's garnet rim analysis to pin down the
sample's peak P–T conditions.

### Pitfalls and tips

- **Accessor calls return *new* DataFrames — they don't change the original.**
  Chain them (`df.oxides.select(...).mineral.end_members(...)`) and keep the
  result in a new variable.
- **The oxygen basis and site model come from the `Mineral` object, not the
  data**: forget the argument (`.mineral.apfu(Cpx)` needs `Cpx`) and you get no
  formula.
- **`.cations(n_oxygens=...)` needs the right oxygen basis per mineral.** If a
  result "looks off", check the basis and the charge labels first — almost
  always the bug is a typo'd column name like `Fe{2+}`.